In [ ]:
!pip install pyspark --quiet
print('succes!!')

succes!!


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year,month,to_date,col,round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
spark=SparkSession.builder\
.appName('Day4_BigData_Sales')\
.config('spark.sql.adaptive.enabled','true')\
.getOrCreate()
print('succes!!')
print("Saprk version:",spark.version)
print('ACTIVE')



succes!!
Saprk version: 4.0.2
ACTIVE


***BRONZE***

In [8]:
df_bronze=spark.read\
.option('header','true')\
.option('inferSchema','true')\
.csv('large_sales_data.csv')
print('======******************BRONZE LAYER->RAW DATA**********************==========')
print("rows:",df_bronze.count())
print("columns:",len(df_bronze.columns))
print("names:",df_bronze.columns)
df_bronze.printSchema()

======******************BRONZE LAYER->RAW DATA**********************==========
rows: 5000
columns: 13
names: ['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [10]:
print('first 5 rows')
print(df_bronze.show(5))
print('last 5 rows')
print(df_bronze.tail(5))
print('basic statistics for numerical columns')
print(df_bronze.select('quantity','unit_price','revenue').describe().show())

first 5 rows
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|   product|   category|quantity|unit_price|revenue|order_date|     city|region|  sales_rep|  payment_method|order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|    1001|  Sneha Reddy|   Monitor|Electronics|      12|     22000| 264000|2023-05-21|   Mumbai|  West|Meera Patel|             UPI|   Delivered|
|    1002| Ramesh Kumar|   Printer|Electronics|      10|     12000| 120000|2023-08-05|    Delhi| North|Anil Sharma|     Credit Card|     Shipped|
|    1003| Rahul Mishra|     Mouse|Accessories|      10|       800|   8000|2023-01-14|Ahmedabad|  West|Meera Patel|Cash on Delivery|     Shipped|
|    1004|   Suresh Rao|    Tablet|Electronics|       5|     32000| 160000|2023-01-04|    Surat|  West| Ravi Ku

In [22]:
from genericpath import isfile
df_bronze.write\
.mode('overwrite')\
.parquet('sales_bronze.parquet')
print('bronze file saved: sales_bronze.parquet')
import os
def get_dir_size(path):
    if os.path.isfile(path):
      return os.path.getsize(path)/1024
    total_size = 0
    for dirpath, dirnames, filenames in os.walk(path):
        for f in filenames:
            total_size += os.path.getsize(os.path.join(dirpath, f))
    return total_size/1024


csv_size=get_dir_size('large_sales_data.csv')
print('csv size:',csv_size)
parquet_size=get_dir_size('sales_bronze.parquet')
print('parquet size:',parquet_size)
reduction=(csv_size-parquet_size)/csv_size * 100
print(f'reduction: {reduction:.2f}% reduced')



bronze file saved: sales_bronze.parquet
csv size: 529.3125
parquet size: 55.09765625
reduction: 89.59% reduced


***SILVER***

In [26]:
df_silver = df_bronze \
    .dropDuplicates() \
    .dropna(subset=['order_id', 'product', 'revenue'])
df_silver = df_silver.withColumn('order_date', F.to_date(F.col('order_date'), 'yyyy-MM-dd'))
df_silver = df_silver.withColumn('order_year', F.year(F.col('order_date')))
df_silver = df_silver.withColumn('order_month', F.month(F.col('order_date')))
df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(F.col('revenue') > 10000, 'high')
     .when(F.col('revenue') > 4000, 'medium')
     .otherwise('low')
)
print(f'Silver layer rows: {df_silver.count()}')
print('New columns added successfully')
df_silver.select('product','order_date', 'order_year', 'order_month', 'revenue_category').show(5)

Silver layer rows: 5000
New columns added successfully
+--------+----------+----------+-----------+----------------+
| product|order_date|order_year|order_month|revenue_category|
+--------+----------+----------+-----------+----------------+
|Keyboard|2023-02-07|      2023|          2|            high|
|  Webcam|2023-01-24|      2023|          1|            high|
| Speaker|2023-04-16|      2023|          4|            high|
|Keyboard|2023-12-21|      2023|         12|          medium|
|  Laptop|2023-08-23|      2023|          8|            high|
+--------+----------+----------+-----------+----------------+
only showing top 5 rows


In [28]:
df_silver.write\
.mode('overwrite')\
.parquet('sales_silver.parquet')
print('silver file saved: sales_silver.parquet')
print('silver parquet fil saved succesfully: sales_silver.parquet')
print('parquet file size:',get_dir_size('sales_silver.parquet'),'kb')
df_verify=spark.read.parquet('sales_silver.parquet')
print('**********************************verify silver layer****************')
print('read back rows:',df_verify.count())
print('read back columns:',len(df_verify.columns))
print('read back names:',df_verify.count())
df_verify.printSchema()

silver file saved: sales_silver.parquet
silver parquet fil saved succesfully: sales_silver.parquet
parquet file size: 59.826171875 kb
**********************************verify silver layer****************
read back rows: 5000
read back columns: 16
read back names: 5000
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)

